# 05 · Comparing the fine-tuned VLMs and the OCR baselines

Merge every `eval_report.json` produced by notebook 03 (VLMs) and notebook 06 (OCR
baselines) into one table, then run the statistical tests before drawing conclusions.

**Prerequisite** — notebook 03 finished for all three models (`qwen`, `internvl`,
`llama_vision`), two reports each. `result/{side}/` should contain:

```
eval_qwen_ft_back.json            eval_qwen_ft_back_preds.jsonl
eval_qwen_zeroshot_back.json      eval_qwen_zeroshot_back_preds.jsonl
eval_internvl_ft_back.json        ...
eval_llama_vision_ft_back.json    ...
```

**Optional but recommended** — notebook 06 adds:

```
eval_paddleocr_back.json          eval_paddleocr_back_preds.jsonl
eval_paddle_vietocr_back.json     eval_paddle_vietocr_back_preds.jsonl
```

Without those two the notebook still runs, but the baseline rows stay empty and the
work cannot show that VLM + QLoRA beats an OCR + rules pipeline on this dataset.

No GPU is needed — this notebook only reads JSON files.

> Why test at all: the test split is ~10% of the data (200 images), so a few points of
> FA can easily be noise. Claiming "model X beats Y" without a confidence interval is
> not supported.

## 1. Mount Drive and inventory the reports

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
import sys
from pathlib import Path

SIDE = 'Front'          # must match notebook 03

DATA_DRIVE = Path('/content/drive/MyDrive/cccd_project/Data')
REPO_DRIVE = DATA_DRIVE / 'label_CCCD'

# One directory per side, or reports collide
LEGACY_DIR = DATA_DRIVE / 'result'
RESULT_DIR = LEGACY_DIR / SIDE.lower()
RESULT_DIR.mkdir(parents=True, exist_ok=True)

# Copy older flat-layout reports; rename is unreliable on Drive
copied = 0
for pattern in (f'eval_*_{SIDE.lower()}.json', f'eval_*_{SIDE.lower()}_preds.jsonl'):
    for p in LEGACY_DIR.glob(pattern):
        target = RESULT_DIR / p.name
        if not target.exists():
            shutil.copy2(p, target)
            copied += 1
if copied:
    print(f'Da copy {copied} file tu result/ -> result/{SIDE.lower()}/ (ban goc giu nguyen)')

sys.path.insert(0, str(REPO_DRIVE))
%cd {REPO_DRIVE}

print('Thu muc report:', RESULT_DIR)
print()
print('Report tim thay:')
found = sorted(RESULT_DIR.glob('eval_*.json'))
for p in found:
    print('  ', p.name)
if not found:
    print('   TRONG - chay notebook 03 truoc.')

print()
print('File du doan (can cho phan thong ke):')
preds = sorted(RESULT_DIR.glob('eval_*_preds.jsonl'))
for p in preds:
    print('  ', p.name)
if not preds:
    print('   TRONG - evaluate.py ban cu khong ghi preds. Chay lai notebook 03.')

Mounted at /content/drive
/content/drive/MyDrive/cccd_project/Data/label_CCCD
Thu muc report: /content/drive/MyDrive/cccd_project/Data/result/front

Report tim thay:
   eval_internvl_ft_front.json
   eval_internvl_zeroshot_front.json
   eval_llama_vision_ft_front.json
   eval_llama_vision_zeroshot_front.json
   eval_paddle_vietocr_front.json
   eval_paddleocr_front.json
   eval_qwen_ft_front.json
   eval_qwen_zeroshot_front.json

File du doan (can cho phan thong ke):
   eval_internvl_ft_front_preds.jsonl
   eval_internvl_zeroshot_front_preds.jsonl
   eval_llama_vision_ft_front_preds.jsonl
   eval_llama_vision_zeroshot_front_preds.jsonl
   eval_paddle_vietocr_front_preds.jsonl
   eval_paddleocr_front_preds.jsonl
   eval_qwen_ft_front_preds.jsonl
   eval_qwen_zeroshot_front_preds.jsonl


## 2. Merge into a comparison table

`scripts/compare_models.py` groups by `run.model_key` and splits by `run.mode`
(zero_shot / fine_tuned), so it only needs the report directory.

In [2]:
!python scripts/compare_models.py --report_dir '{RESULT_DIR}' --report_path '{RESULT_DIR}/model_comparison.json' --markdown_path '{RESULT_DIR}/model_comparison.md'

2026-09-21 09:32:16,971 [INFO] Bỏ qua (không phải eval report): /content/drive/MyDrive/cccd_project/Data/result/front/gen_cost_llama_vision_front.json
2026-09-21 09:32:17,195 [INFO] Bỏ qua (không phải eval report): /content/drive/MyDrive/cccd_project/Data/result/front/model_comparison.json
2026-09-21 09:32:17,195 [INFO] Đã nạp 8 report cho 5 model: internvl, llama_vision, paddle_vietocr, paddleocr, qwen
2026-09-21 09:32:17,527 [INFO] ✅ Đã lưu /content/drive/MyDrive/cccd_project/Data/result/front/model_comparison.json và /content/drive/MyDrive/cccd_project/Data/result/front/model_comparison.md

| Model | Params (B) | FA zero-shot | FA fine-tuned | Δ FA | CER ↓ | micro-F1 | JSON parse | Peak VRAM (MB) | Latency p50/p95 (ms) | Trainable % |
|---|---|---|---|---|---|---|---|---|---|---|
| /content/drive/MyDrive/cccd_project/Data/models/InternVL3_5-2B-HF | 2.3 | 0.6111 | 0.9452 | +0.3341 | 0.0040 | 0.9833 | 1.0000 | 2612.3 | 17174.5 / 18736.6 | 1.1592 |
| /content/drive/MyDrive/cccd_project

In [3]:
from IPython.display import Markdown, display

md_path = RESULT_DIR / 'model_comparison.md'
display(Markdown(md_path.read_text(encoding='utf-8')))

# So sánh VLM fine-tune — trích xuất CCCD

Sinh tự động bởi `scripts/compare_models.py`. Mọi dòng dùng **cùng tập test, cùng prompt/schema, cùng `metrics.evaluate`, cùng greedy decoding**.

## Bảng chính

| Model | Params (B) | FA zero-shot | FA fine-tuned | Δ FA | CER ↓ | micro-F1 | JSON parse | Peak VRAM (MB) | Latency p50/p95 (ms) | Trainable % |
|---|---|---|---|---|---|---|---|---|---|---|
| /content/drive/MyDrive/cccd_project/Data/models/InternVL3_5-2B-HF | 2.3 | 0.6111 | 0.9452 | +0.3341 | 0.0040 | 0.9833 | 1.0000 | 2612.3 | 17174.5 / 18736.6 | 1.1592 |
| /content/drive/MyDrive/cccd_project/Data/models/Llama-3.2-11B-Vision-Instruct | 10.6 | 0.4753 | 0.9437 | +0.4684 | 0.0045 | 0.9823 | 1.0000 | 7970.4 | 19852.7 / 21365.3 | 0.8809 |
| /content/drive/MyDrive/cccd_project/Data/models/Qwen2.5-VL-3B-Instruct | 3.7 | 0.6188 | 0.9375 | +0.3187 | 0.0055 | 0.9788 | 1.0000 | 2731.0 | 19967.3 / 21712.8 | 1.4503 |

> Δ FA = FA(fine-tuned) − FA(zero-shot): đóng góp thuần của QLoRA cho kiến trúc đó.
> Các cột CER / F1 / parse / VRAM / latency lấy từ dòng fine-tuned (nếu có).

## Baseline OCR truyền thống

Không dùng `train.jsonl` ở bất kỳ khâu nào ⇒ **không có cột Δ FA**. Cùng tập test, cùng schema, cùng `metrics.evaluate`, cùng tiền xử lý ảnh 1024px như các VLM.

| Baseline | Detector | Recognizer | FA | CER ↓ | micro-F1 | Trường điền được | Dòng OCR/ảnh | Latency p50/p95 (ms) | Ảnh 1024px |
|---|---|---|---|---|---|---|---|---|---|
| PP-OCR det + VietOCR rec (vgg_transformer) | ppocr_db | vietocr | 0.5147 | 0.5644 | 0.6470 | 0.7199 | 16.2 | 7899.1 / 9721.2 | có |
| PaddleOCR PP-OCR (det+rec, lang=vi) | ppocr_db | ppocr_crnn | 0.4028 | 0.5723 | 0.3767 | 0.7269 | 17.4 | 23376.8 / 33348.8 | có |

## Xếp hạng chung (mọi lần chạy)

| Model / Baseline | Chế độ | FA | CER ↓ | micro-F1 | Latency p50/p95 (ms) |
|---|---|---|---|---|---|
| /content/drive/MyDrive/cccd_project/Data/models/InternVL3_5-2B-HF | fine_tuned | 0.9452 | 0.0040 | 0.9833 | 17174.5 / 18736.6 |
| /content/drive/MyDrive/cccd_project/Data/models/Llama-3.2-11B-Vision-Instruct | fine_tuned | 0.9437 | 0.0045 | 0.9823 | 19852.7 / 21365.3 |
| /content/drive/MyDrive/cccd_project/Data/models/Qwen2.5-VL-3B-Instruct | fine_tuned | 0.9375 | 0.0055 | 0.9788 | 19967.3 / 21712.8 |
| /content/drive/MyDrive/cccd_project/Data/models/Qwen2.5-VL-3B-Instruct | zero_shot | 0.6188 | 0.1660 | 0.7780 | 10275.8 / 11285.7 |
| /content/drive/MyDrive/cccd_project/Data/models/InternVL3_5-2B-HF | zero_shot | 0.6111 | 0.7765 | 0.6673 | 10337.7 / 11772.4 |
| PP-OCR det + VietOCR rec (vgg_transformer) | rule_based | 0.5147 | 0.5644 | 0.6470 | 7899.1 / 9721.2 |
| /content/drive/MyDrive/cccd_project/Data/models/Llama-3.2-11B-Vision-Instruct | zero_shot | 0.4753 | 0.3042 | 0.5775 | 10265.8 / 11437.7 |
| PaddleOCR PP-OCR (det+rec, lang=vi) | rule_based | 0.4028 | 0.5723 | 0.3767 | 23376.8 / 33348.8 |

> `rule_based` = baseline OCR, **chưa từng thấy dữ liệu train**. Đặt cạnh `fine_tuned` để thấy khoảng cách, không phải để kết luận kiến trúc nào hơn.

## FA theo từng trường

Mỗi model lấy bản fine-tuned; baseline OCR lấy bản `rule_based`.

### Mặt trước

| Trường | internvl | llama_vision | paddle_vietocr (rule_based) | paddleocr (rule_based) | qwen |
|---|---|---|---|---|---|
| `so_cccd` | 0.9931 | 0.9861 | 0.8958 | 0.9444 | 0.9861 |
| `ho_va_ten` | 0.8889 | 0.9167 | 0.2083 | 0.0139 | 0.8889 |
| `ngay_sinh` | 1.0000 | 0.9861 | 0.5486 | 0.5764 | 0.9653 |
| `gioi_tinh` | 1.0000 | 1.0000 | 0.6597 | 0.4861 | 1.0000 |
| `quoc_tich` | 0.9931 | 0.9931 | 0.4792 | 0.0000 | 0.9931 |
| `que_quan` | 0.8958 | 0.8889 | 0.2917 | 0.1389 | 0.8819 |
| `noi_thuong_tru` | 0.7639 | 0.7708 | 0.1597 | 0.1319 | 0.7778 |
| `co_gia_tri_den` | 0.9722 | 0.9514 | 0.3889 | 0.3333 | 0.9444 |
| `mat_the` | 1.0000 | 1.0000 | 1.0000 | 1.0000 | 1.0000 |

### Mặt sau

| Trường | internvl | llama_vision | paddle_vietocr (rule_based) | paddleocr (rule_based) | qwen |
|---|---|---|---|---|---|
| `mat_the` | 1.0000 | 1.0000 | 1.0000 | 1.0000 | 1.0000 |

## Trước khi kết luận

Tập test ~10% dữ liệu ⇒ chênh lệch vài % FA có thể là nhiễu. Cần ≥2 seed, bootstrap CI 95% cho FA, và McNemar test trên exact-match từng trường khi so từng cặp model (VLM_COMPARISON_PLAN.md mục 9).


## 3. Bootstrap confidence interval for Field Accuracy

**Resample by image, not by field.** Fields within one image are strongly correlated (a
blurry photo gets a whole batch wrong), so treating each field as an independent
observation would give an artificially narrow interval.

If two intervals overlap, report "no statistically significant difference" rather than
naming a winner.

In [4]:
import json, math, random
from pathlib import Path
from src.utils.metrics import normalize_text

N_BOOT = 2000
SEED   = 42

# Must match BASELINE_MODE in compare_models.py
BASELINE_MODE = 'rule_based'


def load_preds(path):
    """Read a *_preds.jsonl file into a list of {image, pred, gold}."""
    rows = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def score_rows(rows):
    """Return per_image [(n_correct, n_fields)] for bootstrap and flat {(image, field): ok} for McNemar."""
    per_image, flat = [], {}
    for r in rows:
        gold, pred = r.get('gold') or {}, r.get('pred') or {}
        n_ok = 0
        for k, gv in gold.items():
            ok = normalize_text(pred.get(k)) == normalize_text(gv)
            flat[(r['image'], k)] = ok
            n_ok += ok
        per_image.append((n_ok, len(gold)))
    return per_image, flat


def bootstrap_fa(per_image, n_boot=N_BOOT, seed=SEED):
    """95% CI for FA via cluster bootstrap at the image level."""
    rng, N, samples = random.Random(seed), len(per_image), []
    if N == 0:
        return (0.0, 0.0, 0.0)
    tot = sum(x[1] for x in per_image)
    point = sum(x[0] for x in per_image) / tot if tot else 0.0
    for _ in range(n_boot):
        draw = [per_image[rng.randrange(N)] for _ in range(N)]
        t = sum(x[1] for x in draw)
        samples.append(sum(x[0] for x in draw) / t if t else 0.0)
    samples.sort()
    return point, samples[int(0.025 * n_boot)], samples[int(0.975 * n_boot)]


# Baselines load here too, as 'rule_based'
runs = {}
for p in sorted(RESULT_DIR.glob('eval_*_preds.jsonl')):
    report = p.with_name(p.name.replace('_preds.jsonl', '.json'))
    if not report.exists():
        print(f'Bo qua {p.name}: khong thay report tuong ung')
        continue
    run = json.loads(report.read_text(encoding='utf-8')).get('run', {})
    key = (run.get('model_key', p.stem), run.get('mode', 'fine_tuned'))
    per_image, flat = score_rows(load_preds(p))
    runs[key] = {'per_image': per_image, 'flat': flat, 'n_images': len(per_image)}

print(f'{"Model":<16}{"Mode":<12}{"N anh":>7}{"FA":>9}{"CI 95%":>22}')
print('-' * 68)
ci = {}
for key in sorted(runs):
    point, lo, hi = bootstrap_fa(runs[key]['per_image'])
    ci[key] = (point, lo, hi)
    print(f'{key[0]:<16}{key[1]:<12}{runs[key]["n_images"]:>7}'
          f'{point:>8.2%}   [{lo:.2%}, {hi:.2%}]')

# FA/CI need every run on the same images
sizes = {k: v['n_images'] for k, v in runs.items()}
if len(set(sizes.values())) > 1:
    print()
    print('CANH BAO: cac run khong cung so anh ->', sizes)
    print('  McNemar van dung (no giao 2 tap key), nhung FA/CI thi KHONG so sanh')
    print('  truc tiep duoc. Chay lai run bi thieu trên DUNG tap test day du.')

Model           Mode          N anh       FA                CI 95%
--------------------------------------------------------------------
internvl        fine_tuned      144  94.52%   [93.06%, 95.83%]
internvl        zero_shot       144  61.11%   [58.95%, 63.35%]
llama_vision    fine_tuned      144  94.37%   [92.90%, 95.91%]
llama_vision    zero_shot       144  47.53%   [45.22%, 49.69%]
paddle_vietocr  rule_based      144  51.47%   [47.99%, 54.78%]
paddleocr       rule_based      144  40.28%   [37.89%, 42.59%]
qwen            fine_tuned      144  93.75%   [92.13%, 95.22%]
qwen            zero_shot       144  61.88%   [59.65%, 64.12%]


## 4. McNemar test — pairwise comparison

A **paired** test: both models are scored on exactly the same images, so only the
disagreements matter.

- `b` = A correct, B wrong
- `c` = A wrong, B correct
- If the models are equal, `b` and `c` should be close ⇒ exact binomial test with
  p = 0.5 on `b` out of `b + c`

This is far more powerful than comparing two independent rates, because it removes the
variance caused by per-image difficulty.

The cell below runs four groups:

1. Fine-tuned VLMs against each other — which architecture wins?
2. Fine-tuned vs zero-shot of the same model — how much does QLoRA contribute?
3. Fine-tuned VLMs vs the OCR baselines — the direct answer to limitation
   (whether VLM + QLoRA actually beats an OCR + rules pipeline on this dataset).
   Needs the reports from notebook 06.
4. The two baselines against each other — they share a detector, so the difference
   isolates the Vietnamese-specific recognizer.

⚠ The test runs at field level and fields within one image are not fully independent,
so p-values lean optimistic. Read them together with the intervals in section 3.

In [5]:
def mcnemar_exact(flat_a, flat_b, label=''):
    """Exact McNemar over the (image, field) pairs both runs share; returns (b, c, p)."""
    keys = set(flat_a) & set(flat_b)
    # Empty intersection = runs used different images
    if not keys:
        print(f'  !! GIAO RONG {label}: 2 run khong cung tap anh/truong -> bo qua')
        return 0, 0, 1.0
    b = sum(1 for k in keys if flat_a[k] and not flat_b[k])
    c = sum(1 for k in keys if not flat_a[k] and flat_b[k])
    n = b + c
    if n == 0:
        return b, c, 1.0
    m = min(b, c)
    p = 2 * sum(math.comb(n, k) for k in range(m + 1)) / (2 ** n)
    return b, c, min(p, 1.0)


ft_keys = sorted(k for k in runs if k[1] == 'fine_tuned')
print('So tung cap model (deu o che do fine-tuned)')
print()
print(f'{"A vs B":<36}{"A hon":>7}{"B hon":>7}{"p-value":>11}  Ket luan')
print('-' * 88)
for i in range(len(ft_keys)):
    for j in range(i + 1, len(ft_keys)):
        a, bk = ft_keys[i], ft_keys[j]
        b, c, p = mcnemar_exact(runs[a]['flat'], runs[bk]['flat'], f'{a[0]} vs {bk[0]}')
        verdict = ('khac biet CO y nghia (p<0.05)' if p < 0.05
                   else 'KHONG du bang chung de ket luan')
        print(f'{a[0] + " vs " + bk[0]:<36}{b:>7}{c:>7}{p:>11.4f}  {verdict}')

print()
print()
print('Fine-tuned vs Zero-shot cua CUNG mot model (dong gop cua QLoRA)')
print()
print(f'{"Model":<20}{"FT hon":>8}{"ZS hon":>8}{"p-value":>11}  Ket luan')
print('-' * 78)
for mk in sorted({k[0] for k in runs}):
    if (mk, 'fine_tuned') in runs and (mk, 'zero_shot') in runs:
        b, c, p = mcnemar_exact(runs[(mk, 'fine_tuned')]['flat'],
                                runs[(mk, 'zero_shot')]['flat'], f'{mk} ft/zs')
        verdict = 'QLoRA co tac dung ro ret' if p < 0.05 else 'chua du bang chung'
        print(f'{mk:<20}{b:>8}{c:>8}{p:>11.4f}  {verdict}')


# Baselines are not in ft_keys: compare separately
base_keys = sorted(k for k in runs if k[1] == BASELINE_MODE)

print()
print()
print('VLM fine-tuned vs baseline OCR truyen thong  <- cau hoi cua L8')
print()
if not base_keys:
    print('  (Chua co report baseline. Chay notebook 06 truoc, roi chay lai notebook nay.)')
else:
    print(f'{"VLM vs baseline":<40}{"VLM hon":>9}{"OCR hon":>9}{"p-value":>11}  Ket luan')
    print('-' * 96)
    for vk in ft_keys:
        for bk_ in base_keys:
            b, c, p = mcnemar_exact(runs[vk]['flat'], runs[bk_]['flat'],
                                    f'{vk[0]} vs {bk_[0]}')
            if p >= 0.05:
                verdict = 'KHONG du bang chung de ket luan'
            elif b > c:
                verdict = 'VLM hon CO y nghia (p<0.05)'
            else:
                verdict = 'BASELINE hon CO y nghia (p<0.05)'
            print(f'{vk[0] + " vs " + bk_[0]:<40}{b:>9}{c:>9}{p:>11.4f}  {verdict}')

    # Shared detector: isolates the recognizer
    if len(base_keys) > 1:
        print()
        print('Giua 2 pipeline baseline (cung detector => co lap khau nhan dang)')
        print()
        for i in range(len(base_keys)):
            for j in range(i + 1, len(base_keys)):
                a, bk = base_keys[i], base_keys[j]
                b, c, p = mcnemar_exact(runs[a]['flat'], runs[bk]['flat'],
                                        f'{a[0]} vs {bk[0]}')
                verdict = ('khac biet CO y nghia (p<0.05)' if p < 0.05
                           else 'KHONG du bang chung de ket luan')
                print(f'{a[0] + " vs " + bk[0]:<40}{b:>9}{c:>9}{p:>11.4f}  {verdict}')

So tung cap model (deu o che do fine-tuned)

A vs B                                A hon  B hon    p-value  Ket luan
----------------------------------------------------------------------------------------
internvl vs llama_vision                 21     19     0.8746  KHONG du bang chung de ket luan
internvl vs qwen                         39     29     0.2750  KHONG du bang chung de ket luan
llama_vision vs qwen                     38     30     0.3961  KHONG du bang chung de ket luan


Fine-tuned vs Zero-shot cua CUNG mot model (dong gop cua QLoRA)

Model                 FT hon  ZS hon    p-value  Ket luan
------------------------------------------------------------------------------
internvl                 440       7     0.0000  QLoRA co tac dung ro ret
llama_vision             611       4     0.0000  QLoRA co tac dung ro ret
qwen                     425      12     0.0000  QLoRA co tac dung ro ret


VLM fine-tuned vs baseline OCR truyen thong  <- cau hoi cua L8

VLM vs baseline  

## 5. Final table for the report

Combine FA, the confidence intervals and the operational metrics into one markdown table
saved to Drive.

In [6]:
summary = json.loads((RESULT_DIR / 'model_comparison.json').read_text(encoding='utf-8'))

lines = [
    f'# Bang ket qua so sanh VLM - CCCD mat {SIDE}',
    '',
    'Moi dong dung cung tap test, cung prompt/schema, cung `metrics.evaluate`, '
    f'cung greedy decoding. CI 95% bootstrap resample theo anh, {N_BOOT} lan, seed={SEED}.',
    '',
    '| Model | FA zero-shot | FA fine-tuned (CI 95%) | Delta FA | CER | micro-F1 | '
    'JSON parse | VRAM (GB) | p50/p95 (ms) |',
    '|' + '---|' * 9,
]

for mk in sorted(summary):
    modes = summary[mk]
    ft, zs = modes.get('fine_tuned'), modes.get('zero_shot')
    if not ft:
        continue
    point, lo, hi = ci.get((mk, 'fine_tuned'), (ft['field_accuracy'], None, None))
    ci_txt = f'{point:.2%} [{lo:.2%}, {hi:.2%}]' if lo is not None else f'{point:.2%}'
    fa_zs = f'{zs["field_accuracy"]:.2%}' if zs else '-'
    delta = f'{modes["delta_fa"]:+.2%}' if 'delta_fa' in modes else '-'
    lat = ft.get('latency_ms') or {}
    vram = f'{ft["peak_vram_mb"] / 1024:.1f}' if ft.get('peak_vram_mb') else '-'
    lines.append(
        f'| {ft["run"].get("model_id", mk)} | {fa_zs} | {ci_txt} | {delta} | '
        f'{ft["cer"]:.4f} | {ft["f1"]:.2%} | {ft.get("json_parse_rate", 0):.1%} | '
        f'{vram} | {lat.get("p50", "-")} / {lat.get("p95", "-")} |'
    )

lines += [
    '',
    '> Delta FA = FA(fine-tuned) - FA(zero-shot): dong gop thuan cua QLoRA.',
    '> Khoang tin cay chong lan => khong ket luan model nao tot hon.',
    '',
]

# Separate table: Delta FA/VRAM do not apply
base_rows = [(mk, modes[BASELINE_MODE]) for mk, modes in sorted(summary.items())
             if BASELINE_MODE in modes]

if base_rows:
    lines += [
        '## Baseline OCR truyen thong (Stage 3d)',
        '',
        'KHONG co cot Delta FA: baseline khong nhin thay `train.jsonl` o bat ky khau '
        'nao nen khong co cap zero-shot/fine-tuned de tru. Hai pipeline dung CHUNG '
        'detector, nen chenh lech giua chung co lap phan dong gop cua recognizer '
        'chuyen tieng Viet.',
        '',
        '| Pipeline | FA (CI 95%) | CER | micro-F1 | Truong dien duoc | Dong OCR/anh | Engine |',
        '|' + '---|' * 7,
    ]
    for mk, rep in base_rows:
        point, lo, hi = ci.get((mk, BASELINE_MODE),
                               (rep['field_accuracy'], None, None))
        ci_txt = f'{point:.2%} [{lo:.2%}, {hi:.2%}]' if lo is not None else f'{point:.2%}'
        cfg = rep.get('engine_config') or {}
        fill = rep.get('field_fill_rate')
        fill_txt = f'{fill:.1%}' if isinstance(fill, (int, float)) else '-'
        rows_txt = rep.get('ocr_lines_per_image', '-')
        run = rep.get('run') or {}
        # Tag ablations so they are not read as headline
        flags = []
        if run.get('canonicalize'):
            flags.append('canonicalize')
        if not run.get('image_max_side'):
            flags.append('full-res')
        name = mk + (f' _({"+".join(flags)})_' if flags else '')
        lines.append(
            f'| {name} | {ci_txt} | {rep["cer"]:.4f} | {rep["f1"]:.2%} | '
            f'{fill_txt} | {rows_txt} | {cfg.get("engine", "-")} |'
        )
    lines += [
        '',
        '> "Truong dien duoc" = `field_fill_rate`: ty le truong parser dien ra MOT CAI GI DO, '
        'bat ke dung sai. Doc kem FA de tach "OCR khong doc ra" khoi "doc ra nhung sai".',
        '> Dong nao co nhan _(canonicalize)_ hoac _(full-res)_ la ABLATION, khong phai so headline.',
        '',
    ]
else:
    lines += [
        '> Chua co report baseline OCR. Chay notebook 06 de bo sung - thieu no thi '
        'han che L8 van con nguyen.',
        '',
    ]

out = RESULT_DIR / f'final_comparison_{SIDE.lower()}.md'
out.write_text(chr(10).join(lines), encoding='utf-8')
print('Da luu ->', out)
print()
display(Markdown(chr(10).join(lines)))

Da luu -> /content/drive/MyDrive/cccd_project/Data/result/front/final_comparison_front.md



# Bang ket qua so sanh VLM - CCCD mat Front

Moi dong dung cung tap test, cung prompt/schema, cung `metrics.evaluate`, cung greedy decoding. CI 95% bootstrap resample theo anh, 2000 lan, seed=42.

| Model | FA zero-shot | FA fine-tuned (CI 95%) | Delta FA | CER | micro-F1 | JSON parse | VRAM (GB) | p50/p95 (ms) |
|---|---|---|---|---|---|---|---|---|
| /content/drive/MyDrive/cccd_project/Data/models/InternVL3_5-2B-HF | 61.11% | 94.52% [93.06%, 95.83%] | +33.41% | 0.0040 | 98.33% | 100.0% | 2.6 | 17174.5 / 18736.6 |
| /content/drive/MyDrive/cccd_project/Data/models/Llama-3.2-11B-Vision-Instruct | 47.53% | 94.37% [92.90%, 95.91%] | +46.84% | 0.0045 | 98.23% | 100.0% | 7.8 | 19852.7 / 21365.3 |
| /content/drive/MyDrive/cccd_project/Data/models/Qwen2.5-VL-3B-Instruct | 61.88% | 93.75% [92.13%, 95.22%] | +31.87% | 0.0055 | 97.88% | 100.0% | 2.7 | 19967.3 / 21712.8 |

> Delta FA = FA(fine-tuned) - FA(zero-shot): dong gop thuan cua QLoRA.
> Khoang tin cay chong lan => khong ket luan model nao tot hon.

## Baseline OCR truyen thong (Stage 3d)

KHONG co cot Delta FA: baseline khong nhin thay `train.jsonl` o bat ky khau nao nen khong co cap zero-shot/fine-tuned de tru. Hai pipeline dung CHUNG detector, nen chenh lech giua chung co lap phan dong gop cua recognizer chuyen tieng Viet.

| Pipeline | FA (CI 95%) | CER | micro-F1 | Truong dien duoc | Dong OCR/anh | Engine |
|---|---|---|---|---|---|---|
| paddle_vietocr | 51.47% [47.99%, 54.78%] | 0.5644 | 64.70% | 72.0% | 16.2 | - |
| paddleocr | 40.28% [37.89%, 42.59%] | 0.5723 | 37.67% | 72.7% | 17.4 | - |

> "Truong dien duoc" = `field_fill_rate`: ty le truong parser dien ra MOT CAI GI DO, bat ke dung sai. Doc kem FA de tach "OCR khong doc ra" khoi "doc ra nhung sai".
> Dong nao co nhan _(canonicalize)_ hoac _(full-res)_ la ABLATION, khong phai so headline.


## 6. What the test set is made of: real vs synthetic, templates, split leakage

Every number above averages over one test split as if its images were interchangeable.
They are not. The corpus was built in two batches, and notebook 01 shows the seam: a
one-off cell marked **every record with index ≥ 505** as reviewed in a single pass
(`mark_reviewed_colab`, `MIN_INDEX = 505`, 1098 of 1602 back records). A batch that
needs no human review is a batch whose labels are known by construction — synthetic
cards rendered from a template, not photographs.

That changes how three things must be read:

1. **A headline FA is a weighted average of two populations.** If the synthetic images
   are easier, the headline overstates what the model does on real cards, by exactly as
   much as synthetic images dominate the split.
2. **Templates ("phôi") are shared, images are not.** `prepare_dataset.split_groups`
   uses `group_key = filename stem` (`prepare_dataset.py:19-21`), so *the image* never
   crosses splits — but a template, a layout, or a whole set of field values can appear
   in train **and** test. Then "unseen test image" is true at the file level and false
   at the level the model actually learns.
3. **Field-value duplicates are worse than template duplicates.** A shared blank card is
   a shared background; a shared `so_cccd`/`ngay_cap` tuple is the answer itself sitting
   in the training set.

Three cells, all CPU:

- **6a** decides real vs synthetic from the pixels and checks that against the 505 seam.
- **6b** re-scores every run separately on each group, with the same cluster bootstrap.
- **6c** counts templates by perceptual hash and cross-tabulates them, and the gold
  values, against train/val/test.

> 6a reads every original image once from Drive, which is slow (a few minutes for ~1600
> files). The other two cells are fast.

In [7]:
# 6a. Anh nao la anh CHUP THAT, anh nao la anh RENDER tong hop?
#
# Khong tin san ranh gioi 505 cua notebook 01: cell nay do lai tu chinh file anh
# (kich thuoc, dinh dang, EXIF, dung luong, do nhieu tan so cao) roi moi doi chieu.
# Anh chup co nhieu cam bien + artifact JPEG, kich thuoc lung tung, thuong con EXIF.
# Anh render tu cung mot phoi thi kich thuoc GIONG HET nhau va gan nhu khong nhieu.
# Tien the tinh luon dHash cua tung anh de cell 6c dem phoi ma khong doc lai Drive.
import json
import re
import unicodedata
from collections import Counter, defaultdict

import numpy as np
from PIL import Image
from tqdm.auto import tqdm

from src.utils.metrics import safe_parse

IDX_CUT = 505                       # tu notebook 01, cell mark_reviewed_colab
DATASET_DIR = DATA_DRIVE / 'dataset' / SIDE
IMAGE_ROOT = DATA_DRIVE / SIDE
AUG_RE = re.compile(r'_aug\d+$')
IMG_EXT = {'.png', '.jpg', '.jpeg', '.webp', '.bmp'}


def orig_stem(image_path):
    """Stem anh GOC: bo duoi _aug0/_aug1 ma prepare_dataset.py them vao."""
    return AUG_RE.sub('', Path(str(image_path)).stem)


def img_index(stem):
    m = re.search(r'(\d+)$', stem)
    return int(m.group(1)) if m else -1


def strip_tones(text):
    """Bo dau, giong squash() cua field_parser: d/D khong phan ra duoi NFD."""
    text = text.replace('đ', 'd').replace('Đ', 'D')
    return ''.join(c for c in unicodedata.normalize('NFD', text)
                   if not unicodedata.combining(c)).lower()


# --- 1. Anh goc nao thuoc split nao (ban augment gop vao anh goc cua no) ---
split_of, gold_of, n_aug_of = {}, {}, Counter()
for split in ('train', 'val', 'test'):
    path = DATASET_DIR / f'{split}.jsonl'
    if not path.exists():
        print(f'   THIEU {path} - chay notebook 02 muc 4 truoc')
        continue
    for line in open(path, encoding='utf-8'):
        if not line.strip():
            continue
        rec = json.loads(line)
        stem_full = Path(rec['image']).stem
        stem = orig_stem(stem_full)
        if AUG_RE.search(stem_full):
            n_aug_of[stem] += 1
            continue
        split_of[stem] = split
        gold_of[stem] = safe_parse(rec['conversations'][1]['value'])

print(f'Anh goc theo split: {dict(Counter(split_of.values()))}')
print(f'Ban augment: {sum(n_aug_of.values())} anh tren {len(n_aug_of)} anh goc; '
      f'split cua chung = {sorted({split_of.get(s) for s in n_aug_of})} '
      f'(phai chi co "train")')

# --- 2. Thuoc tinh cua tung anh goc ---
on_disk = {}
for p in IMAGE_ROOT.iterdir():
    if p.suffix.lower() in IMG_EXT:
        on_disk.setdefault(p.stem, p)

props, missing = {}, []
for stem in tqdm(sorted(split_of), desc='Doc anh goc'):
    path = on_disk.get(stem)
    if path is None:
        missing.append(stem)
        continue
    try:
        with Image.open(path) as im:
            size, fmt = im.size, im.format
            has_exif = bool(im.getexif())
            gray = np.asarray(im.convert('L').resize((256, 160), Image.Resampling.BILINEAR),
                              dtype=np.float32)
    except Exception as exc:
        print(f'   Bo qua {stem}: {type(exc).__name__}')
        continue
    lap = (4 * gray[1:-1, 1:-1] - gray[:-2, 1:-1] - gray[2:, 1:-1]
           - gray[1:-1, :-2] - gray[1:-1, 2:])
    # dHash 64 bit, tinh luon o day de cell 6c khong phai doc lai Drive mot luot nua
    small = np.asarray(Image.fromarray(gray.astype(np.uint8))
                       .resize((9, 8), Image.Resampling.LANCZOS), dtype=np.int16)
    props[stem] = {'idx': img_index(stem), 'size': size, 'fmt': fmt, 'exif': has_exif,
                   'kb': path.stat().st_size / 1024, 'noise': float(lap.std()),
                   'dhash': (small[:, 1:] > small[:, :-1]).reshape(-1).astype(np.uint8)}
if missing:
    print(f'   {len(missing)} stem trong jsonl khong co file anh (vd {missing[:3]})')

# --- 3. Dac diem 2 lo, chia theo ranh gioi 505 ---
groups = defaultdict(list)
for pr in props.values():
    groups[f'idx < {IDX_CUT}' if pr['idx'] < IDX_CUT else f'idx >= {IDX_CUT}'].append(pr)

print(f'\n{"=" * 96}\nDAC DIEM 2 LO ANH\n{"=" * 96}')
print(f'{"lo":12s} {"n":>5s} {"kich thuoc hay gap":>26s} {"#size":>6s} '
      f'{"KB p50":>8s} {"nhieu p50":>10s} {"co EXIF":>9s}')
print('-' * 96)
for name in (f'idx < {IDX_CUT}', f'idx >= {IDX_CUT}'):
    g = groups.get(name, [])
    if not g:
        continue
    sizes = Counter(pr['size'] for pr in g)
    top, n_top = sizes.most_common(1)[0]
    print(f'{name:12s} {len(g):5d} {f"{top[0]}x{top[1]} ({n_top / len(g):.0%})":>26s} '
          f'{len(sizes):6d} {np.median([pr["kb"] for pr in g]):8.1f} '
          f'{np.median([pr["noise"] for pr in g]):10.2f} '
          f'{sum(pr["exif"] for pr in g) / len(g):8.0%}')
print('\nDoc bang: lo render thi #size nho (vai kich thuoc), nhieu thap, khong EXIF.')
print('Neu 2 dong GIONG nhau thi gia thuyet real/synthetic KHONG dung -> dung dung tiep.')

# --- 4. Do lai ranh gioi tu chinh anh, khong lay san 505 ---
# Ba dac trung DOC LAP, moi cai tu do diem cat rieng. Mot dac trung "thoai hoa"
# (moi anh deu giong nhau) se cho acc = 100% mot cach gia tao, nen phai bat rieng:
# no khong tach duoc gi ca, khong phai no tach hoan hao.
items = sorted(props.items(), key=lambda kv: kv[1]['idx'])
idxs = np.array([pr['idx'] for _, pr in items])

hi = groups.get(f'idx >= {IDX_CUT}', [])
modal_size = Counter(pr['size'] for pr in hi).most_common(1)[0][0] if hi else None
noise_mid = float(np.median([pr['noise'] for pr in props.values()]))
kb_mid = float(np.median([pr['kb'] for pr in props.values()]))

FEATURES = {
    'kich thuoc = modal': np.array([pr['size'] == modal_size for _, pr in items]),
    'nhieu < trung vi': np.array([pr['noise'] < noise_mid for _, pr in items]),
    'dung luong < trung vi': np.array([pr['kb'] < kb_mid for _, pr in items]),
    'khong co EXIF': np.array([not pr['exif'] for _, pr in items]),
}


def scan_cut(flags):
    """(diem cat tot nhat, do chinh xac, co thoai hoa khong)."""
    share = float(flags.mean())
    # < 2% hoac > 98% mot phia => dac trung nay khong mang thong tin
    if share < 0.02 or share > 0.98:
        return None, float('nan'), True
    best, acc = None, -1.0
    for cut in np.unique(idxs):
        a = float(((idxs >= cut) == flags).mean())
        if a > acc:
            best, acc = int(cut), a
    return best, acc, False


print(f'\n{"=" * 96}\nDO LAI RANH GIOI TU ANH (notebook 01 dung {IDX_CUT})\n{"=" * 96}')
print(f'{"dac trung":24s} {"ty le True":>11s} {"diem cat":>9s} {"xep dung":>9s}  ghi chu')
print('-' * 96)
votes = []
for name, flags in FEATURES.items():
    cut, acc, degenerate = scan_cut(flags)
    if degenerate:
        note = 'THOAI HOA - moi anh nhu nhau, khong tach duoc gi'
        print(f'{name:24s} {flags.mean():11.1%} {"-":>9s} {"-":>9s}  {note}')
        continue
    note = 'ok' if acc >= 0.97 else 'tach khong sach'
    votes.append((name, cut, acc))
    print(f'{name:24s} {flags.mean():11.1%} {cut:9d} {acc:9.2%}  {note}')

# Diem cat khong nam tren luoi dao: no roi vao khoang TRONG giua 2 lo.
lo_side = idxs[idxs < IDX_CUT]
hi_side = idxs[idxs >= IDX_CUT]
if lo_side.size and hi_side.size and hi_side.min() - lo_side.max() > 1:
    print(f'\nKhong anh nao co chi so trong [{lo_side.max() + 1}, {hi_side.min() - 1}] '
          f'=> moi diem cat trong khoang nay cho KET QUA Y HET.')
elif lo_side.size and hi_side.size:
    print(f'\nHai lo lien nhau o chi so {lo_side.max()} / {hi_side.min()} - khong co '
          f'khoang trong, nen xe dich diem cat 1 don vi la doi nhan 1 anh.')

# --- 5. Doi chieu DOC LAP: tu vung cua nhan mat sau ---
# Lo that dung ky hieu bien ban cong an: "C:2 cm", "0,5cm", "seo cham", "seo thang".
# Lo tong hop dung tu vung sinh may: "cach 1cm", "vet thuong cu o ...", "bot ... o ...".
# Day la bang chung nhin vao NHAN, hoan toan doc lap voi 4 dac trung nhin vao PIXEL.
vocab_cross = None
if SIDE.lower() == 'back':
    RE_REAL = re.compile(r'c\s*:\s*\d|,\d\s*cm|seo cham|seo thang')
    RE_SYN = re.compile(r'\bcach\b|vet thuong cu|\bbot \w+ o\b|khong co dac diem')
    tab = Counter()
    for stem, pr in props.items():
        plain = strip_tones((gold_of.get(stem) or {}).get('dac_diem_nhan_dang') or '')
        lo = 'real' if pr['idx'] < IDX_CUT else 'synth'
        if not plain:
            style = '(rong)'
        elif RE_REAL.search(plain):
            style = 'kieu bien ban'
        elif RE_SYN.search(plain):
            style = 'kieu sinh may'
        else:
            style = 'khac'
        tab[(lo, style)] += 1
    print(f'\n{"=" * 96}\nDOI CHIEU DOC LAP - tu vung `dac_diem_nhan_dang`\n{"=" * 96}')
    for lo in ('real', 'synth'):
        row = {s: c for (l, s), c in sorted(tab.items()) if l == lo}
        print(f'  {lo:6s} {row}')
    vocab_cross = tab[('real', 'kieu sinh may')] + tab[('synth', 'kieu bien ban')]
    print(f'  So ca lan sang nhom kia: {vocab_cross}')

# --- 6. VERDICT: co duoc dung nhan real/synthetic khong? ---
reasons = []
if not votes:
    reasons.append('ca 4 dac trung pixel deu thoai hoa - anh hai lo khong khac gi nhau')
else:
    best_name, best_cut, best_acc = max(votes, key=lambda v: v[2])
    if best_acc < 0.97:
        reasons.append(f'dac trung tot nhat ({best_name}) chi xep dung {best_acc:.1%}')
    if abs(best_cut - IDX_CUT) > 60:
        reasons.append(f'diem cat do lai ({best_cut}) lech xa {IDX_CUT} cua notebook 01')
    agree = [v for v in votes if abs(v[1] - IDX_CUT) <= 60 and v[2] >= 0.97]
    if len(votes) > 1 and not agree:
        reasons.append('khong dac trung nao vua chinh xac vua dong y voi moc 505')
if vocab_cross is not None and vocab_cross > 0.02 * len(props):
    reasons.append(f'tu vung nhan lan sang nhom kia {vocab_cross} ca (> 2%)')

PROVENANCE_TRUSTED = not reasons
print(f'\n{"=" * 96}')
if PROVENANCE_TRUSTED:
    print('VERDICT: DAT. Nhan real/synthetic theo chi so anh la co can cu do duoc.')
    if votes:
        print(f'  Dac trung pixel manh nhat: {best_name} -> cat tai {best_cut} '
              f'({best_acc:.2%}), khop moc {IDX_CUT} cua notebook 01.')
    if vocab_cross == 0:
        print('  Tu vung nhan tach sach tuyet doi (0 ca lan) - bang chung doc lap thu hai.')
else:
    print('VERDICT: KHONG DAT. Dung dung bang FA tach real/synthetic o muc 10.')
    for r in reasons:
        print(f'  - {r}')
    print('  Cell 6b van chay duoc nhung ket qua cua no KHONG dan chung duoc.')
print('=' * 96)

# --- 7. Ghi ra de cac cell sau dung lai ---
PROVENANCE = {stem: ('real' if pr['idx'] < IDX_CUT else 'synth') for stem, pr in props.items()}
prov_path = RESULT_DIR / f'provenance_{SIDE.lower()}.json'
prov_path.write_text(json.dumps(
    {'side': SIDE, 'idx_cut': IDX_CUT, 'tin_duoc': PROVENANCE_TRUSTED,
     'ly_do_khong_tin': reasons,
     'diem_cat_do_lai': {n: {'cut': c, 'acc': round(a, 4)} for n, c, a in votes},
     'tu_vung_lan_nhom': vocab_cross,
     'n_real': sum(v == 'real' for v in PROVENANCE.values()),
     'n_synth': sum(v == 'synth' for v in PROVENANCE.values()),
     'nhan': PROVENANCE}, ensure_ascii=False, indent=2), encoding='utf-8')

print(f'\nTy le real/synth toan bo mat {SIDE}:')
for split in ('train', 'val', 'test'):
    sub = Counter(PROVENANCE[s] for s in split_of if split_of[s] == split and s in PROVENANCE)
    tot = sum(sub.values()) or 1
    print(f'  {split:6s} n={tot:5d}  real={sub["real"]:4d} ({sub["real"] / tot:5.1%})  '
          f'synth={sub["synth"]:4d} ({sub["synth"] / tot:5.1%})')
print(f'\nDa luu -> {prov_path}')


Anh goc theo split: {'train': 1148, 'val': 143, 'test': 144}
Ban augment: 2298 anh tren 1148 anh goc; split cua chung = ['train'] (phai chi co "train")


Doc anh goc:   0%|          | 0/1435 [00:00<?, ?it/s]


DAC DIEM 2 LO ANH
lo               n         kich thuoc hay gap  #size   KB p50  nhieu p50   co EXIF
------------------------------------------------------------------------------------------------
idx < 505      126              640x640 (76%)     11    556.8      31.47       0%
idx >= 505    1309              640x640 (77%)     61    517.5      30.30       0%

Doc bang: lo render thi #size nho (vai kich thuoc), nhieu thap, khong EXIF.
Neu 2 dong GIONG nhau thi gia thuyet real/synthetic KHONG dung -> dung dung tiep.

DO LAI RANH GIOI TU ANH (notebook 01 dung 505)
dac trung                 ty le True  diem cat  xep dung  ghi chu
------------------------------------------------------------------------------------------------
kich thuoc = modal             76.5%         8    76.59%  tach khong sach
nhieu < trung vi               50.0%      1279    52.68%  tach khong sach
dung luong < trung vi          50.0%       903    52.47%  tach khong sach
khong co EXIF                 100.0%         

In [8]:
# 6b. Cham lai MOI run, tach rieng nhom real va nhom synthetic.
#
# Cung normalize_text, cung bootstrap cum theo anh, cung SEED nhu muc 3 -> chi khac
# o tap anh duoc tinh. Bo `mat_the` vi no suy tu ten file chu khong phai doc anh
# (cccd_schema.infer_side_from_filename), giu lai se cong diem khong cua ai.
assert 'PROVENANCE' in globals(), 'Chay cell 6a truoc.'
assert 'bootstrap_fa' in globals(), 'Chay cell muc 3 truoc (noi dinh nghia bootstrap_fa).'
if not globals().get('PROVENANCE_TRUSTED', False):
    print('!! 6a KET LUAN KHONG DAT - bang duoi day khong dan chung duoc, '
          'xem ly do o cell 6a.\n')

DROP_FIELDS = {'mat_the'}


def per_image_by_group(rows, want):
    """[(so truong dung, tong so truong)] cho rieng nhom `want`."""
    out = []
    for r in rows:
        if PROVENANCE.get(orig_stem(r['image'])) != want:
            continue
        gold = {k: v for k, v in (r.get('gold') or {}).items() if k not in DROP_FIELDS}
        pred = r.get('pred') or {}
        n_ok = sum(1 for k, gv in gold.items()
                   if normalize_text(pred.get(k)) == normalize_text(gv))
        out.append((n_ok, len(gold)))
    return out


def per_field_by_group(rows, want):
    hit, tot = Counter(), Counter()
    for r in rows:
        if PROVENANCE.get(orig_stem(r['image'])) != want:
            continue
        pred = r.get('pred') or {}
        for k, gv in (r.get('gold') or {}).items():
            if k in DROP_FIELDS:
                continue
            tot[k] += 1
            hit[k] += normalize_text(pred.get(k)) == normalize_text(gv)
    return {k: hit[k] / tot[k] for k in tot}


preds_files = sorted(RESULT_DIR.glob('eval_*_preds.jsonl'))
unknown = set()
split_fa = {}

print(f'Da bo khoi FA: {sorted(DROP_FIELDS)} - chi cham tren truong noi dung.')
print(f'\n{"run":24s} {"n real":>7s} {"FA real":>8s} {"CI 95% real":>18s} '
      f'{"n syn":>6s} {"FA syn":>8s} {"CI 95% syn":>18s} {"syn - real":>11s}')
print('-' * 108)
for p in preds_files:
    report_path = p.with_name(p.name.replace('_preds.jsonl', '.json'))
    if not report_path.exists():
        continue
    rows = load_preds(p)
    unknown |= {orig_stem(r['image']) for r in rows if orig_stem(r['image']) not in PROVENANCE}
    real, syn = per_image_by_group(rows, 'real'), per_image_by_group(rows, 'synth')
    if not real or not syn:
        print(f'{p.stem[:24]:24s} (mot trong hai nhom rong -> bo qua)')
        continue
    fr, lr, hr = bootstrap_fa(real)
    fs, ls, hs = bootstrap_fa(syn)
    label = p.stem.replace('eval_', '').replace(f'_{SIDE.lower()}_preds', '')
    split_fa[label] = {'n_real': len(real), 'fa_real': fr, 'ci_real': [lr, hr],
                       'n_synth': len(syn), 'fa_synth': fs, 'ci_synth': [ls, hs]}
    print(f'{label:24s} {len(real):7d} {fr:8.2%} {f"[{lr:.2%}, {hr:.2%}]":>18s} '
          f'{len(syn):6d} {fs:8.2%} {f"[{ls:.2%}, {hs:.2%}]":>18s} {fs - fr:+11.2%}')

if unknown:
    print(f'\n   {len(unknown)} anh trong preds khong co nhan provenance '
          f'(vd {sorted(unknown)[:3]}) - chung bi bo khoi CA HAI nhom.')

# Truong nao tao ra khoang cach? Chi truong van ban tu do moi dang ke.
print(f'\n{"=" * 100}\nACCURACY TUNG TRUONG - real / synth\n{"=" * 100}')
for p in preds_files:
    if not p.with_name(p.name.replace('_preds.jsonl', '.json')).exists():
        continue
    rows = load_preds(p)
    pr, ps = per_field_by_group(rows, 'real'), per_field_by_group(rows, 'synth')
    if not pr or not ps:
        continue
    label = p.stem.replace('eval_', '').replace(f'_{SIDE.lower()}_preds', '')
    cells = '  '.join(f'{k}: {pr.get(k, 0):6.2%}/{ps.get(k, 0):6.2%}'
                      for k in sorted(set(pr) | set(ps)))
    print(f'{label:24s} {cells}')

print('\nDoc bang: cot "syn - real" duong nghia la nhom tong hop DE hon, tuc so headline')
print('o muc 2-5 dang duoc keo len boi ty le anh tong hop trong tap test.')
print('Khoang tin cay cua nhom real rong hon han vi n nho - dung so sanh diem voi diem.')

fa_path = RESULT_DIR / f'fa_by_provenance_{SIDE.lower()}.json'
fa_path.write_text(json.dumps(split_fa, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'\nDa luu -> {fa_path}')

!! 6a KET LUAN KHONG DAT - bang duoi day khong dan chung duoc, xem ly do o cell 6a.

Da bo khoi FA: ['mat_the'] - chi cham tren truong noi dung.

run                       n real  FA real        CI 95% real  n syn   FA syn         CI 95% syn  syn - real
------------------------------------------------------------------------------------------------------------
internvl_ft                   15   95.83%  [90.83%, 100.00%]    129   93.60%   [91.96%, 95.16%]      -2.23%
internvl_zeroshot             15   71.67%   [65.00%, 78.33%]    129   67.93%   [65.41%, 70.45%]      -3.74%
llama_vision_ft               15   96.67%  [92.50%, 100.00%]    129   93.31%   [91.57%, 95.06%]      -3.35%
llama_vision_zeroshot         15   55.00%   [44.17%, 64.17%]    129   53.10%   [50.39%, 55.72%]      -1.90%
paddle_vietocr                15   55.83%   [47.50%, 63.33%]    129   44.19%   [40.21%, 47.97%]     -11.65%
paddleocr                     15   36.67%   [30.83%, 43.33%]    129   32.36%   [29.65%, 35.17%]  

In [9]:
# 6c. Co bao nhieu PHOI (template), va phoi/gia tri co dung chung giua cac split khong?
#
# split_groups() lay group_key = ten file (prepare_dataset.py:19-21) nen moi ANH chi
# nam o dung mot split. Dieu do KHONG chan:
#   - cung mot phoi xuat hien o ca train lan test  -> model thuoc bo cuc, khong doc chu
#   - cung mot bo gia tri gold xuat hien o ca hai  -> dap an nam san trong train
# Cell nay do ca hai. Phoi do bang dHash: cung phoi thi anh gan nhu trung nhau tru
# vung chu, nen khoang cach Hamming rat nho.
assert 'props' in globals(), 'Chay cell 6a truoc.'
assert all('dhash' in pr for pr in props.values()), \
    'props chua co khoa "dhash" - chay lai cell 6a (ban moi co tinh dHash).'

HAM_T = 6          # nguong Hamming coi la CUNG phoi; xem bang quet nguong ben duoi

names = sorted(props)
pos_of = {n: i for i, n in enumerate(names)}
prov_by_name = [PROVENANCE[n] for n in names]
bits = np.array([props[n]['dhash'] for n in names], dtype=np.int32)   # (N, 64)
ham = bits @ (1 - bits).T + (1 - bits) @ bits.T
np.fill_diagonal(ham, 64)

nn = ham.min(axis=1)
print(f'Khoang cach Hamming toi lang gieng gan nhat (N={len(names)}):')
for lo, hi in ((0, 1), (1, 3), (3, 7), (7, 13), (13, 65)):
    n = int(((nn >= lo) & (nn < hi)).sum())
    print(f'  [{lo:2d}, {hi:2d})  {n:5d}  {n / len(names):6.1%}  {"#" * int(60 * n / len(names))}')


def count_clusters(threshold):
    """Union-find tren canh Hamming <= threshold."""
    parent = np.arange(len(names))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    ii, jj = np.nonzero(np.triu(ham <= threshold, 1))
    for a, b in zip(ii.tolist(), jj.tolist()):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb
    return [find(i) for i in range(len(names))]


print('\nSo cum theo nguong (de thay ket qua co on dinh voi nguong khong):')
print(f'{"nguong":>7s} {"so cum":>7s} {"cum lon nhat":>13s}')
for t in (0, 2, 4, 6, 8, 10, 12):
    lab = count_clusters(t)
    sizes = Counter(lab)
    print(f'{t:7d} {len(sizes):7d} {max(sizes.values()):13d}')

labels = count_clusters(HAM_T)
clusters = defaultdict(list)
for name, lab in zip(names, labels):
    clusters[lab].append(name)

n_real_cl = len({lab for lab, prov in zip(labels, prov_by_name) if prov == 'real'})
n_syn_cl = len({lab for lab, prov in zip(labels, prov_by_name) if prov == 'synth'})
print(f'\n{"=" * 92}\nPHOI (nguong Hamming <= {HAM_T})\n{"=" * 92}')
print(f'  Tong so cum        : {len(clusters)} tren {len(names)} anh')
print(f'  Cum chua anh real  : {n_real_cl}   (real: {prov_by_name.count("real")} anh)')
print(f'  Cum chua anh synth : {n_syn_cl}   (synth: {prov_by_name.count("synth")} anh)')
print(f'  Cum co > 1 anh     : {sum(1 for v in clusters.values() if len(v) > 1)}')

print(f'\n  10 cum lon nhat:')
print(f'  {"n anh":>6s} {"train":>6s} {"val":>4s} {"test":>5s} {"real/synth":>11s}  vi du')
for lab, members in sorted(clusters.items(), key=lambda kv: -len(kv[1]))[:10]:
    sp = Counter(split_of[m] for m in members)
    pv = Counter(PROVENANCE[m] for m in members)
    mix = f'{pv["real"]}/{pv["synth"]}'
    print(f'  {len(members):6d} {sp["train"]:6d} {sp["val"]:4d} {sp["test"]:5d} '
          f'{mix:>11s}  {members[0]}')

# --- Phoi co vat qua split khong? ---
spanning = [lab for lab, m in clusters.items() if len({split_of[x] for x in m}) > 1]
test_names = [n for n in names if split_of[n] == 'test']
test_leak = [n for n in test_names
             if any(split_of[m] == 'train' for m in clusters[labels[pos_of[n]]])]
print(f'\n  Cum vat qua >= 2 split : {len(spanning)} / {len(clusters)}')
print(f'  Anh TEST co cung phoi voi it nhat 1 anh TRAIN: '
      f'{len(test_leak)} / {len(test_names)} ({len(test_leak) / max(len(test_names), 1):.1%})')

# --- Ro ri o muc GIA TRI, nang hon ro ri o muc phoi ---
def gold_key(stem):
    g = gold_of.get(stem) or {}
    return tuple(sorted((k, normalize_text(v)) for k, v in g.items() if k not in DROP_FIELDS))


key_splits = defaultdict(set)
key_members = defaultdict(list)
for n in names:
    key_splits[gold_key(n)].add(split_of[n])
    key_members[gold_key(n)].append(n)

dup_keys = {k: v for k, v in key_members.items() if len(v) > 1}
test_val_leak = [n for n in test_names if 'train' in (key_splits[gold_key(n)] - {'test'})]
print(f'\n{"=" * 92}\nRO RI O MUC GIA TRI GOLD\n{"=" * 92}')
print(f'  Bo gia tri gold duy nhat : {len(key_splits)} / {len(names)} anh')
print(f'  Bo gia tri bi lap        : {len(dup_keys)} bo, phu '
      f'{sum(len(v) for v in dup_keys.values())} anh')
print(f'  Anh TEST co bo gia tri Y HET mot anh TRAIN: '
      f'{len(test_val_leak)} / {len(test_names)} ({len(test_val_leak) / max(len(test_names), 1):.1%})')
if test_val_leak:
    print(f'    vi du: {test_val_leak[:5]}')

# Truong dinh danh nhat cua tung mat
ID_FIELD = 'so_cccd' if SIDE.lower() == 'front' else 'ngay_cap'
id_members = defaultdict(list)
for n in names:
    val = normalize_text((gold_of.get(n) or {}).get(ID_FIELD))
    if val:
        id_members[val].append(n)
id_dup = {k: v for k, v in id_members.items() if len(v) > 1}
id_leak = [n for n in test_names
           if any(split_of[m] == 'train'
                  for m in id_members.get(normalize_text((gold_of.get(n) or {}).get(ID_FIELD)), []))]
print(f'\n  Theo rieng truong `{ID_FIELD}`:')
print(f'    gia tri bi lap : {len(id_dup)} / {len(id_members)}')
print(f'    anh TEST trung `{ID_FIELD}` voi mot anh TRAIN: '
      f'{len(id_leak)} / {len(test_names)} ({len(id_leak) / max(len(test_names), 1):.1%})')

out = {'ham_threshold': HAM_T, 'n_images': len(names), 'n_clusters': len(clusters),
       'n_clusters_spanning_splits': len(spanning),
       'test_share_template_with_train': len(test_leak),
       'test_share_gold_with_train': len(test_val_leak),
       'test_share_id_field_with_train': len(id_leak), 'id_field': ID_FIELD,
       'n_test': len(test_names)}
tpl_path = RESULT_DIR / f'templates_{SIDE.lower()}.json'
tpl_path.write_text(json.dumps(out, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'\nDa luu -> {tpl_path}')
print('\nDoc ket qua:')
print('  · So cum << so anh  => it phoi, dung chung nhieu. Con so nay la "so phoi".')
print('  · "Anh TEST cung phoi voi anh TRAIN" cao la BINH THUONG voi anh tong hop, nhung')
print('    phai noi ro trong bai: tap test khong do duoc kha nang tong quat sang phoi la.')
print('  · "Anh TEST co bo gia tri y het anh TRAIN" moi la ro ri that: dap an nam san')
print('    trong train, model chi can nho. Con so nay > 0 thi phai bao cao rieng.')

Khoang cach Hamming toi lang gieng gan nhat (N=1435):
  [ 0,  1)    496   34.6%  ####################
  [ 1,  3)     91    6.3%  ###
  [ 3,  7)    278   19.4%  ###########
  [ 7, 13)    458   31.9%  ###################
  [13, 65)    112    7.8%  ####

So cum theo nguong (de thay ket qua co on dinh voi nguong khong):
 nguong  so cum  cum lon nhat
      0    1069            70
      2    1007           252
      4     910           252
      6     785           252
      8     557           252
     10     349           624
     12     176           913

PHOI (nguong Hamming <= 6)
  Tong so cum        : 785 tren 1435 anh
  Cum chua anh real  : 92   (real: 126 anh)
  Cum chua anh synth : 730   (synth: 1309 anh)
  Cum co > 1 anh     : 215

  10 cum lon nhat:
   n anh  train  val  test  real/synth  vi du
     252    215   18    19      21/231  cccd_front_100
      23     14    4     5        0/23  cccd_front_1328
      22     19    2     1        1/21  cccd_front_1023
      11      7    1  

## 7. Checklist before writing the conclusion

**VLM branch**

- [ ] All 6 runs present (3 models × {zero-shot, fine-tuned})? The table in section 2 has a "missing" list.
- [ ] Is `trainable_params_pct` the same order of magnitude across models? A large gap means the LoRA budgets are not equivalent and must be explained.
- [ ] Is `image_tokens` from the notebook 02 probe comparable across models? This is the control for how much visual information each model receives.
- [ ] Do the confidence intervals overlap? If so, write "no statistically significant difference", not "model X is better".
- [ ] Same number of epochs for all three models?
- [ ] At least 2 seeds? If not, state it in the limitations.
- [ ] Section 6a: does the real/synthetic split hold up on the pixels (one modal image size, no EXIF, low noise on one side only)? If it does not, drop every real/synthetic claim.
- [ ] Section 6b: is the headline FA quoted alongside the real-only FA? A gap means the headline is carried by the synthetic images and must not be reported on its own.
- [ ] Section 6c: how many templates, and how many test images share one with train? State it — the test split measures generalisation *within* those templates, not beyond them.
- [ ] Section 6c: is `test_share_gold_with_train` zero? If not, those images are memorisation, not extraction, and belong in a separate row.

**OCR baseline branch**

- [ ] Both pipelines (`paddleocr`, `paddle_vietocr`) present? Run notebook 06.
- [ ] Are the headline numbers the run with **both ablation flags off**? Check `run.canonicalize == false` and `run.image_max_side == 1024`. Rows tagged _(canonicalize)_ or _(full-res)_ in section 5 are ablations.
- [ ] Does `run.det_limit_side_len` equal `run.image_max_side`? A mismatch means the detector saw a different number of pixels than intended.
- [ ] Did the baseline run on the same `test.jsonl` as the VLMs? Section 3 warns on a size mismatch, section 4 prints `GIAO RONG` when the image sets differ.
- [ ] Read `field_fill_rate` alongside FA: low FA with high fill means **misread**, low FA with low fill means **not read at all** or a field-splitting bug. These lead to different conclusions.
- [ ] Claiming "VLM beats the baseline" needs the p-value from group 3 in section 4; an FA gap alone is not enough.